# Fish Audio Drama-3 — Server GPU trên Colab

Tổng hợp giọng nói **Fish Audio Drama 3 (44.1 kHz)** trên GPU Colab, máy ở nhà
gọi lên qua API (`/health`, `/voice`, `/tts`, `/tts_batch`).

## Chạy lần lượt từ trên xuống

| ô | việc | lần đầu | lần sau |
|---|---|---|---|
| 1 | kiểm tra GPU | vài giây | vài giây |
| 2 | clone repo + tải engine | ~1 phút | vài giây |
| 3 | tải model (~15.9 GiB) | 5–15 phút | vài giây nếu còn phiên |
| 4 | bật server | ~1–2 phút | ~1–2 phút |
| 5 | mở đường hầm, lấy URL + key | ~20 giây | ~20 giây |
| 7 | giữ phiên sống, tự dựng lại khi hỏng | chạy liên tục | chạy liên tục |

Xong ô 5 thì chép URL + API key về máy mình.

## Chỉ có 2 thứ cần chỉnh (ô 4)

- `VRAM_MODE`: để **`auto`**. Nếu bị hết VRAM thì chọn `2.8gb` (chậm hơn nhưng dùng ít VRAM nhất).
- `THREADS`: để **`0`** = tự lấy theo số nhân CPU của Colab (thường là 2).

Model chạy thẳng bản gốc `drama-3-backbone-8bit-experts-Q2_K.gguf`. Bước nén INT4 đã
bỏ vì công cụ nén cần GLIBC mới hơn Colab nên không chạy được.


In [ ]:
# ── 1. Kiểm tra GPU, RAM & Ổ đĩa ────────────────────────────────────────────
import shutil, subprocess

if not shutil.which("nvidia-smi"):
    raise SystemExit(
        "Runtime này KHÔNG có GPU.\n"
        "Sửa: Runtime -> Change runtime type -> T4 GPU (hoặc L4/A100), rồi chạy lại."
    )

print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print("=== RAM & Ổ đĩa (/content) ===")
print(subprocess.run("free -h && echo '---' && df -h /content", shell=True,
                     capture_output=True, text=True).stdout)


In [ ]:
# ── 2. Clone repo + Tải engine audio.cpp CUDA 12.8 ──────────────────────────
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/GrayPham/submodulevoice.git"
BRANCH   = "master"
APP      = Path("/content/submodulevoice")
ENG_DIR  = Path("/content/drama3_engine")
MOD_DIR  = Path("/content/drama3_models")

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    if p.returncode != 0:
        raise SystemExit(f"Thất bại (rc={p.returncode}): {cmd}")

if APP.exists():
    sh(f"git -C {APP} fetch --depth 1 origin {BRANCH} && "
       f"git -C {APP} reset --hard origin/{BRANCH}")
else:
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {APP}")

sh("pip install -q httpx numpy soundfile soxr")
sh(f"python remote/prepare_drama3_int4.py --engine-dir {ENG_DIR} --models-dir {MOD_DIR} --engine-only --no-compress",
   cwd=str(APP))


In [ ]:
# ── 3. Tải model Drama-3 (~15.9 GiB, có sẵn thì bỏ qua) ─────────────────────
from pathlib import Path

MOD_DIR.mkdir(parents=True, exist_ok=True)
have = [g for g in MOD_DIR.rglob("*.gguf") if g.stat().st_size > 12 * 1024**3]
if have:
    print(f"[có sẵn] {have[0].name} ({have[0].stat().st_size / (1024**3):.2f} GiB)")
else:
    sh(f"python remote/prepare_drama3_int4.py --engine-dir {ENG_DIR} "
       f"--models-dir {MOD_DIR} --workers 8 --no-compress", cwd=str(APP))


In [ ]:
#@title 4. Bật Server Fish Drama-3 { display-mode: "form" }
#@markdown `auto` là đủ. Hết VRAM thì chọn `2.8gb`. `THREADS` = 0 nghĩa là tự theo số nhân CPU.
#@markdown `CPU_EXPERT_LAYERS` = -1 là theo profile (T4: 24). Muốn nhanh hơn thì thử 16 rồi 12:
#@markdown ít tầng hơn = nhanh hơn, tốn VRAM hơn (nếu OOM thì tăng lại), và giảm RAM máy.
VRAM_MODE = "auto"  #@param ["auto", "2.8gb", "t4", "2.3gb", "3.3gb", "gpu"]
PORT = 8770  #@param {type:"integer"}
THREADS = 0  #@param {type:"integer"}
CPU_EXPERT_LAYERS = -1  #@param {type:"integer"}

import json, os, secrets, subprocess, time, urllib.request

THREADS = THREADS or os.cpu_count() or 2
print(f"VRAM_MODE={VRAM_MODE} | THREADS={THREADS} | CPU_EXPERT_LAYERS={CPU_EXPERT_LAYERS}")

LOG = "/content/drama3_server.log"
KEY_FILE = "/content/drama3_api_key.txt"
if os.path.exists(KEY_FILE):
    API_KEY = open(KEY_FILE).read().strip()
else:
    API_KEY = secrets.token_urlsafe(12)
    open(KEY_FILE, "w").write(API_KEY)

subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
subprocess.run("pkill -f audiocpp_server || true", shell=True)
time.sleep(1)

env = dict(os.environ, PYTHONUNBUFFERED="1")
with open(LOG, "wb") as f:
    subprocess.Popen(
        [
            "python", "remote/drama3_server.py",
            "--port", str(PORT),
            "--key", API_KEY,
            "--engine-dir", str(ENG_DIR),
            "--models-dir", str(MOD_DIR),
            "--vram-mode", VRAM_MODE,
            "--threads", str(THREADS),
            "--cpu-expert-layers", str(CPU_EXPERT_LAYERS),
        ],
        stdout=f, stderr=subprocess.STDOUT, env=env, cwd=str(APP),
        start_new_session=True,
    )

def _mem():
    try:
        mi = {}
        for line in open("/proc/meminfo"):
            k, v = line.split(":", 1)
            mi[k] = int(v.split()[0]) // 1024
        ram = f"{mi['MemTotal'] - mi['MemAvailable']}/{mi['MemTotal']}"
    except Exception:
        ram = "?"
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
    return f"RAM {ram} MiB | VRAM {gpu}"

log_pos = 0
for i in range(240):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=3) as r:
            h = json.load(r)
        print(json.dumps(h, ensure_ascii=False, indent=2))
        print(f"\nFISH DRAMA-3 SERVER SẴN SÀNG  (44.1 kHz | mode={h.get('vram_mode')})")
        print(f"API key: {API_KEY}")
        break
    except Exception:
        # In tiếp log của server để thấy nó đang làm gì (không còn "im lặng" khi nạp model).
        try:
            with open(LOG, "rb") as lf:
                lf.seek(log_pos)
                chunk = lf.read()
                log_pos = lf.tell()
            if chunk:
                print(chunk.decode("utf-8", "replace"), end="", flush=True)
        except Exception:
            pass
        if i % 10 == 0:
            print(f"[{i * 2}s] đang nạp... {_mem()}", flush=True)
        time.sleep(2)
else:
    print(open(LOG, encoding="utf-8", errors="replace").read()[-4000:])
    raise SystemExit("Server không lên sau 480s — xem log phía trên.")


In [ ]:
# ── 5. Mở đường hầm Cloudflared, lấy URL + API key ──────────────────────────
import re, subprocess, time
from pathlib import Path

if not Path("/content/cloudflared").exists():
    subprocess.run(
        "wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/"
        "releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared",
        shell=True, check=True,
    )

subprocess.run("pkill -f cloudflared || true", shell=True)
subprocess.Popen(
    f"/content/cloudflared tunnel --url http://127.0.0.1:{PORT} "
    "--no-autoupdate > /content/cloudflared.log 2>&1",
    shell=True, start_new_session=True,
)

URL = None
for _ in range(40):
    time.sleep(2)
    m = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com",
        Path("/content/cloudflared.log").read_text(errors="replace"),
    )
    if m:
        URL = m.group(0)
        break

if not URL:
    print(Path("/content/cloudflared.log").read_text(errors="replace")[-3000:])
    raise SystemExit("Không lấy được URL đường hầm cloudflared")

print("=" * 72)
print("  CHÉP VỀ MÁY MÌNH (FISH DRAMA-3 INT4)")
print("=" * 72)
print(f"  URL      {URL}")
print(f"  API key  {API_KEY}")
print("=" * 72)
print("\nLệnh gọi từ máy bạn (tự động nhận 44.1 kHz):\n")
print(f"  python remote/client.py --url {URL} --key {API_KEY} \\")
print("      --script scripts/kichban_dai.txt \\")
print("      --ref path/to/giong_mau.wav --ref-text 'lời thoại chuẩn của file mẫu' \\")
print("      --concurrency 1 --batch 4 -o output/drama3_out.wav")


In [ ]:
#@title 6. Tự kiểm tra kết nối & Test giọng mẫu (tuỳ chọn) { display-mode: "form" }
#@markdown Điền đường dẫn file WAV mẫu + lời thoại trên Colab để test `/voice` + `/tts`.
#@markdown Để trống `TEST_REF_WAV` sẽ chỉ kiểm tra `/health` qua local + Cloudflared.
TEST_REF_WAV = ""  #@param {type:"string"}
TEST_REF_TEXT = ""  #@param {type:"string"}
TEST_TEXT = "Xin chào mọi người! [chuckles] Đây là bài kiểm tra giọng nói Fish Audio Drama 3 chạy trên Colab."  #@param {type:"string"}

import base64, json, os, urllib.request
from IPython.display import Audio, display

for label, base in (("127.0.0.1", f"http://127.0.0.1:{PORT}"), ("Cloudflared", URL)):
    with urllib.request.urlopen(base + "/health", timeout=20) as r:
        h = json.load(r)
    print(f"[health {label}] ok={h.get('ok')} | model={h.get('model')} | sr={h.get('sample_rate')}")

if TEST_REF_WAV.strip() and os.path.isfile(TEST_REF_WAV.strip()):
    hdrs = {"Content-Type": "application/json", "X-API-Key": API_KEY}
    wav_b64 = base64.b64encode(open(TEST_REF_WAV.strip(), "rb").read()).decode("ascii")
    req_v = urllib.request.Request(
        f"{URL}/voice",
        data=json.dumps({"name": "test_ref", "text": TEST_REF_TEXT.strip(), "wav_b64": wav_b64}).encode(),
        headers=hdrs, method="POST",
    )
    with urllib.request.urlopen(req_v, timeout=180) as r:
        vid = json.load(r)["voice_id"]
    print("Đăng ký voice_id =", vid)

    req_t = urllib.request.Request(
        f"{URL}/tts",
        data=json.dumps({"text": TEST_TEXT, "voice_id": vid}).encode(),
        headers=hdrs, method="POST",
    )
    with urllib.request.urlopen(req_t, timeout=300) as r:
        out_wav = r.read()
        print(f"Xong: audio={r.headers.get('X-Audio-Seconds')}s | synth={r.headers.get('X-Synth-Seconds')}s")
    open("/content/drama3_test.wav", "wb").write(out_wav)
    display(Audio("/content/drama3_test.wav"))
else:
    print("Bỏ qua test tổng hợp vì chưa nhập TEST_REF_WAV (Fish Drama-3 cần 1 file giọng mẫu).")


In [ ]:
# ── 7. Giữ phiên sống + HIỆN LOG RENDER (ĐỂ Ô NÀY CHẠY LIÊN TỤC) ───────────
# Cứ TICK giây: in dòng mới của log server (bắt đầu / xong từng đoạn, VRAM...) và,
# khi đang render, in "ĐANG RENDER mm:ss". Mỗi CHECK_EVERY giây có thêm dòng tổng hợp.
import json, os, re, subprocess, time, urllib.request
from datetime import datetime
from pathlib import Path

# Nếu Colab ngắt rồi nối lại mà kernel mất biến, tự khôi phục từ file/log trên đĩa.
# Khi đó chỉ cần chạy lại ĐÚNG ô này (miễn là server còn chạy), không cần chạy lại ô 1-5.
# Giữ TAB Colab mở và đừng để máy ngủ: Colab chỉ thu hồi phiên khi trình duyệt rảnh.
PORT = globals().get("PORT", 8770)
LOG = globals().get("LOG", "/content/drama3_server.log")
APP = globals().get("APP", Path("/content/submodulevoice"))
ENG_DIR = globals().get("ENG_DIR", Path("/content/drama3_engine"))
MOD_DIR = globals().get("MOD_DIR", Path("/content/drama3_models"))
VRAM_MODE = globals().get("VRAM_MODE", "auto")
THREADS = globals().get("THREADS", os.cpu_count() or 2)
CPU_EXPERT_LAYERS = globals().get("CPU_EXPERT_LAYERS", -1)
if "API_KEY" not in globals():
    API_KEY = open("/content/drama3_api_key.txt").read().strip()
if "URL" not in globals():
    _cf = Path("/content/cloudflared.log")
    _m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                   _cf.read_text(errors="replace") if _cf.exists() else "")
    URL = _m.group(0) if _m else None
    print(f"URL khôi phục từ log: {URL}")

CHECK_EVERY = 60
TICK = 5
last_served = -1
busy_since = None
next_full = 0.0
next_tunnel = 0.0
LOG_POS = os.path.getsize(LOG) if os.path.exists(LOG) else 0

def health():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

def tunnel_alive() -> bool:
    if subprocess.run("pgrep -f cloudflared", shell=True, capture_output=True).returncode != 0:
        return False
    if not URL:
        return False
    try:
        with urllib.request.urlopen(URL + "/health", timeout=15) as r:
            return r.status == 200
    except Exception:
        return False

def tail_log():
    global LOG_POS
    try:
        if os.path.getsize(LOG) < LOG_POS:
            LOG_POS = 0
        with open(LOG, "rb") as f:
            f.seek(LOG_POS)
            data = f.read()
            LOG_POS = f.tell()
    except FileNotFoundError:
        return
    for line in data.decode("utf-8", "replace").splitlines():
        if line.strip():
            print(f"   | {line[:220]}", flush=True)

print(f"Theo dõi: log mỗi {TICK}s, tổng hợp mỗi {CHECK_EVERY}s. Để ô này chạy liên tục.\n", flush=True)
while True:
    now = time.time()
    ts = datetime.now().strftime("%H:%M:%S")
    tail_log()
    h = health()
    if h is None:
        busy_since = None
        print(f"[{ts}] SERVER CHẾT -> dựng lại...", flush=True)
        subprocess.run("pkill -f audiocpp_server || true", shell=True)
        with open(LOG, "ab") as f:
            subprocess.Popen(
                [
                    "python", "remote/drama3_server.py",
                    "--port", str(PORT), "--key", API_KEY,
                    "--engine-dir", str(ENG_DIR), "--models-dir", str(MOD_DIR),
                    "--vram-mode", VRAM_MODE, "--threads", str(THREADS),
                    "--cpu-expert-layers", str(CPU_EXPERT_LAYERS),
                ],
                stdout=f, stderr=subprocess.STDOUT,
                env=dict(os.environ, PYTHONUNBUFFERED="1"),
                cwd=str(APP), start_new_session=True,
            )
        time.sleep(45)
        tail_log()
        print(f"[{ts}] {'đã lên lại' if health() else 'VẪN CHƯA LÊN, xem ' + LOG}", flush=True)
    else:
        if h["busy"]:
            busy_since = busy_since or now
            m, s = divmod(int(now - busy_since), 60)
            print(
                f"[{ts}] ĐANG RENDER {m:02d}:{s:02d} | xong {h['served']} | "
                f"hàng đợi {h['queued']} | lỗi {h['failed']}",
                flush=True,
            )
        else:
            busy_since = None
        if now >= next_full:
            d = h["served"] - last_served if last_served >= 0 else h["served"]
            last_served = h["served"]
            next_full = now + CHECK_EVERY
            gb = h.get("gpu", {})
            print(
                f"[{ts}] === busy={h['busy']} | queue={h['queued']} | done={h['served']} (+{d}) | "
                f"err={h['failed']} | VRAM free={gb.get('vram_free_mib')} MiB ===",
                flush=True,
            )

    if now >= next_tunnel:
        next_tunnel = now + CHECK_EVERY
        if not tunnel_alive():
            print(f"[{ts}] ĐƯỜNG HẦM ĐỨT -> mở lại...", flush=True)
            subprocess.run("pkill -f cloudflared || true", shell=True)
            time.sleep(2)
            subprocess.Popen(
                f"/content/cloudflared tunnel --url http://127.0.0.1:{PORT} "
                "--no-autoupdate > /content/cloudflared.log 2>&1",
                shell=True, start_new_session=True,
            )
            time.sleep(20)
            m = re.search(
                r"https://[a-z0-9-]+\.trycloudflare\.com",
                Path("/content/cloudflared.log").read_text(errors="replace"),
            )
            if m and m.group(0) != URL:
                URL = m.group(0)
                print(f"[{ts}] URL MỚI: {URL}  (dán lại vào app)", flush=True)

    time.sleep(TICK)
